# 02 - Extract Admin Monitoring semantic models

Reads the selected gold-layer tables from all four Microsoft-managed Admin Monitoring semantic models by using the Power BI Execute Queries API.

- `baseline` mode captures the current rolling history and initializes the archive.
- `daily` mode adds newly observed rows, updates last-seen dates, and retires rows no longer present.
- Every source table is count-checked, schema-versioned, and audited.
- API size guards fail explicitly rather than silently accepting truncated data. Oversized tables should be routed to the Admin REST API fallback.


In [ ]:
run_mode = "baseline"
source_workspace_name = "Admin monitoring"
target_workspace_name = "YOUR_TARGET_WORKSPACE"
lakehouse_name = "AdminMonitoringArchive"
data_schema = "admin_monitoring"
control_schema = "admin_monitoring_control"
max_rows_per_query = 90000
max_values_per_query = 900000


In [ ]:
import hashlib
import json
import re
import time
import uuid
from datetime import date, datetime, timezone

import requests
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType


In [ ]:
def api_request(method, url, audience="pbi", json_body=None, extra_headers=None, attempts=4):
    headers = {"Authorization": f"Bearer {notebookutils.credentials.getToken(audience)}"}
    if json_body is not None:
        headers["Content-Type"] = "application/json"
    if extra_headers:
        headers.update(extra_headers)

    for attempt in range(1, attempts + 1):
        response = requests.request(method, url, headers=headers, json=json_body, timeout=180)
        if response.status_code != 429:
            return response
        if attempt == attempts:
            return response
        time.sleep(int(response.headers.get("Retry-After", "10")))
    raise RuntimeError("Unreachable API retry state.")


def require_success(response, operation, expected=(200,)):
    if response.status_code not in expected:
        raise RuntimeError(
            f"{operation} returned HTTP {response.status_code}: {response.text[:3000]}"
        )
    return response


def power_bi_get(path):
    response = api_request("GET", f"https://api.powerbi.com/v1.0/myorg/{path.lstrip('/')}")
    return require_success(response, f"Power BI GET {path}").json()


def execute_dax(workspace_id, dataset_id, query):
    response = api_request(
        "POST",
        f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{dataset_id}/executeQueries",
        json_body={
            "queries": [{"query": query}],
            "serializerSettings": {"includeNulls": True},
        },
    )
    payload = require_success(response, "Power BI Execute Queries").json()
    results = payload.get("results") or []
    tables = results[0].get("tables") or [] if results else []
    return tables[0].get("rows") or [] if tables else []


def fabric_get(path):
    response = api_request(
        "GET",
        f"https://api.fabric.microsoft.com/v1/{path.lstrip('/')}",
        extra_headers={"x-ms-fabric-skill": "spark-cli"},
    )
    return require_success(response, f"Fabric GET {path}").json()


In [ ]:
def exact_match(items, name_key, expected_name, item_type):
    matches = [
        item for item in items
        if str(item.get(name_key, "")).casefold() == expected_name.casefold()
    ]
    if len(matches) != 1:
        raise LookupError(
            f'Expected one {item_type} named "{expected_name}"; found {len(matches)}.'
        )
    return matches[0]


def dax_table(table_name):
    return "'" + table_name.replace("'", "''") + "'"


def dax_column(table_name, column_name):
    return f"{dax_table(table_name)}[{column_name.replace(chr(93), chr(93) * 2)}]"


def result_column_name(key):
    text = str(key)
    if "[" in text and text.endswith("]"):
        return text.rsplit("[", 1)[1][:-1]
    return text


def sanitize_identifier(value):
    identifier = re.sub(r"[^a-z0-9]+", "_", str(value).casefold()).strip("_")
    if not identifier:
        identifier = "unnamed"
    if identifier[0].isdigit():
        identifier = f"c_{identifier}"
    return identifier


def unique_column_map(source_names):
    mapping = {}
    used = set()
    for source_name in source_names:
        base = sanitize_identifier(source_name)
        candidate = base
        suffix = 2
        while candidate in used:
            candidate = f"{base}_{suffix}"
            suffix += 1
        mapping[source_name] = candidate
        used.add(candidate)
    return mapping


def string_value(value):
    if value is None:
        return None
    if isinstance(value, (dict, list)):
        return json.dumps(value, sort_keys=True, separators=(",", ":"))
    return str(value)


def rows_to_spark(rows, column_metadata, model_name, table_name, snapshot_date, extracted_at):
    metadata_by_name = {str(row.get("[Name]")): row for row in column_metadata}
    source_names = [
        str(row.get("[Name]")) for row in column_metadata
        if row.get("[Type]") in (None, "Data", "Calculated")
    ]
    for row in rows:
        for key in row:
            name = result_column_name(key)
            if name not in source_names:
                source_names.append(name)
    column_map = unique_column_map(source_names)
    schema = StructType([StructField(column_map[name], StringType(), True) for name in source_names])

    records = []
    for row in rows:
        by_name = {result_column_name(key): value for key, value in row.items()}
        records.append(tuple(string_value(by_name.get(name)) for name in source_names))
    dataframe = spark.createDataFrame(records, schema)

    for source_name, target_name in column_map.items():
        data_type = str(metadata_by_name.get(source_name, {}).get("[DataType]", "Text"))
        if data_type == "Integer":
            dataframe = dataframe.withColumn(target_name, F.col(target_name).cast("long"))
        elif data_type in ("Decimal", "Double"):
            dataframe = dataframe.withColumn(target_name, F.col(target_name).cast("double"))
        elif data_type == "Currency":
            dataframe = dataframe.withColumn(target_name, F.col(target_name).cast("decimal(38,6)"))
        elif data_type == "Boolean":
            dataframe = dataframe.withColumn(target_name, F.col(target_name).cast("boolean"))
        elif data_type == "Date":
            dataframe = dataframe.withColumn(target_name, F.to_timestamp(F.col(target_name)))

    source_columns = list(column_map.values())
    hash_parts = [F.coalesce(F.col(name).cast("string"), F.lit("<NULL>")) for name in source_columns]
    dataframe = dataframe.withColumn("_row_hash", F.sha2(F.concat_ws("||", *hash_parts), 256))
    dataframe = dataframe.groupBy("_row_hash").agg(
        *[F.first(F.col(name), ignorenulls=False).alias(name) for name in source_columns],
        F.count(F.lit(1)).cast("long").alias("_duplicate_count"),
    )
    dataframe = (
        dataframe
        .withColumn("_source_model", F.lit(model_name))
        .withColumn("_source_table", F.lit(table_name))
        .withColumn("_snapshot_date_utc", F.to_date(F.lit(snapshot_date.isoformat())))
        .withColumn("_first_seen_date", F.to_date(F.lit(snapshot_date.isoformat())))
        .withColumn("_last_seen_date", F.to_date(F.lit(snapshot_date.isoformat())))
        .withColumn("_is_current", F.lit(True))
        .withColumn("_extracted_at_utc", F.lit(extracted_at).cast("timestamp"))
    )
    return dataframe, column_map


def upsert_history(dataframe, path, snapshot_date, mode):
    if mode == "baseline" or not DeltaTable.isDeltaTable(spark, path):
        dataframe.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)
        return

    dataframe.limit(0).write.format("delta").mode("append").option("mergeSchema", "true").save(path)
    target = DeltaTable.forPath(spark, path)
    (
        target.alias("target")
        .merge(dataframe.alias("source"), "target._row_hash = source._row_hash")
        .whenMatchedUpdate(set={
            "_last_seen_date": "source._last_seen_date",
            "_is_current": "true",
            "_duplicate_count": "source._duplicate_count",
            "_extracted_at_utc": "source._extracted_at_utc",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    if mode == "daily":
        target.update(
            condition=F.col("_last_seen_date") < F.to_date(F.lit(snapshot_date.isoformat())),
            set={"_is_current": F.lit(False)},
        )


In [ ]:
def scalar_count(workspace_id, dataset_id, table_name, filter_expression=None):
    table = dax_table(table_name)
    expression = table if filter_expression is None else f"FILTER({table}, {filter_expression})"
    rows = execute_dax(workspace_id, dataset_id, f'EVALUATE ROW("__RowCount", COUNTROWS({expression}))')
    if len(rows) != 1:
        raise RuntimeError(f"Count query for {table_name} returned {len(rows)} rows.")
    value = next(iter(rows[0].values()), None)
    return 0 if value is None else int(value)


def extract_table_rows(workspace_id, dataset_id, table_name, column_metadata, source_count):
    column_count = max(1, len(column_metadata))
    if source_count <= max_rows_per_query and source_count * column_count <= max_values_per_query:
        rows = execute_dax(workspace_id, dataset_id, f"EVALUATE {dax_table(table_name)}")
        if len(rows) != source_count:
            raise RuntimeError(
                f"{table_name} count mismatch: source={source_count}, extracted={len(rows)}."
            )
        return rows

    metadata_names = {str(row.get("[Name]")) for row in column_metadata}
    if table_name != "Audit" or "Creation date" not in metadata_names:
        raise RuntimeError(
            f"{table_name} exceeds safe Execute Queries limits "
            f"({source_count} rows x {column_count} columns). Route this table to the Admin REST API fallback."
        )

    date_rows = execute_dax(
        workspace_id,
        dataset_id,
        f'EVALUATE DISTINCT(SELECTCOLUMNS({dax_table(table_name)}, "__Date", {dax_column(table_name, "Creation date")}))',
    )
    all_rows = []
    for date_row in date_rows:
        raw_date = next(iter(date_row.values()), None)
        if raw_date is None:
            continue
        parsed = datetime.fromisoformat(str(raw_date).replace("Z", "+00:00"))
        filter_expression = (
            f'{dax_column(table_name, "Creation date")} = DATE({parsed.year}, {parsed.month}, {parsed.day})'
        )
        day_count = scalar_count(workspace_id, dataset_id, table_name, filter_expression)
        if day_count > max_rows_per_query or day_count * column_count > max_values_per_query:
            raise RuntimeError(
                f"Audit date {parsed.date()} exceeds safe Execute Queries limits. "
                "Route Audit extraction to the Admin Activity Events API fallback."
            )
        day_rows = execute_dax(
            workspace_id,
            dataset_id,
            f"EVALUATE FILTER({dax_table(table_name)}, {filter_expression})",
        )
        if len(day_rows) != day_count:
            raise RuntimeError(
                f"Audit {parsed.date()} count mismatch: source={day_count}, extracted={len(day_rows)}."
            )
        all_rows.extend(day_rows)
    if len(all_rows) != source_count:
        raise RuntimeError(f"Audit partition total mismatch: source={source_count}, extracted={len(all_rows)}.")
    return all_rows


In [ ]:
if run_mode not in {"baseline", "daily"}:
    raise ValueError("run_mode must be either 'baseline' or 'daily'.")

context = notebookutils.runtime.context
target_workspace_id = str(context.get("currentWorkspaceId") or "")
current_workspace_name = str(context.get("currentWorkspaceName") or "")
if not target_workspace_id or current_workspace_name.casefold() != target_workspace_name.casefold():
    raise RuntimeError(
        f'This notebook must run in "{target_workspace_name}". Current workspace: "{current_workspace_name}".'
    )

lakehouses = fabric_get(f"workspaces/{target_workspace_id}/lakehouses").get("value", [])
lakehouse = exact_match(lakehouses, "displayName", lakehouse_name, "Lakehouse")
lakehouse_id = str(lakehouse["id"])
endpoint = notebookutils.conf.get("trident.onelake.endpoint").removeprefix("https://").removeprefix("http://").rstrip("/")
table_root = f"abfss://{target_workspace_id}@{endpoint}/{lakehouse_id}/Tables"
control_root = f"{table_root}/{control_schema}"
data_root = f"{table_root}/{data_schema}"

manifest_path = f"{control_root}/table_manifest"
if not DeltaTable.isDeltaTable(spark, manifest_path):
    raise RuntimeError("Table manifest is missing. Run Step 01 first.")
manifest = spark.read.format("delta").load(manifest_path).collect()

groups = power_bi_get("groups?$top=5000").get("value", [])
source_workspace = exact_match(groups, "name", source_workspace_name, "Power BI workspace")
source_workspace_id = str(source_workspace["id"])
datasets = power_bi_get(f"groups/{source_workspace_id}/datasets").get("value", [])
model_names = sorted({row["semantic_model"] for row in manifest})
models = {
    name: exact_match(datasets, "name", name, "semantic model")
    for name in model_names
}

columns_by_model = {}
for model_name, model in models.items():
    columns_by_model[model_name] = execute_dax(
        source_workspace_id, str(model["id"]), "EVALUATE INFO.VIEW.COLUMNS()"
    )

run_id = str(uuid.uuid4())
snapshot_date = datetime.now(timezone.utc).date()
audit_records = []
schema_records = []
failures = []

for manifest_row in manifest:
    model_name = manifest_row["semantic_model"]
    table_name = manifest_row["source_table"]
    model_id = str(models[model_name]["id"])
    target_table = f"{sanitize_identifier(model_name)}__{sanitize_identifier(table_name)}"
    target_path = f"{data_root}/{target_table}"
    started = datetime.now(timezone.utc)
    source_count = None
    extracted_count = None
    schema_hash = None
    error_message = None
    status = "Succeeded"

    try:
        table_columns = [
            row for row in columns_by_model[model_name]
            if str(row.get("[Table]")) == table_name
        ]
        if not table_columns:
            raise RuntimeError(f"No column metadata returned for {model_name}.{table_name}.")
        columns_json = json.dumps(table_columns, sort_keys=True, separators=(",", ":"))
        schema_hash = hashlib.sha256(columns_json.encode("utf-8")).hexdigest()
        source_count = scalar_count(source_workspace_id, model_id, table_name)
        rows = extract_table_rows(
            source_workspace_id, model_id, table_name, table_columns, source_count
        )
        extracted_count = len(rows)
        extracted_at = datetime.now(timezone.utc)
        dataframe, column_map = rows_to_spark(
            rows, table_columns, model_name, table_name, snapshot_date, extracted_at
        )
        upsert_history(dataframe, target_path, snapshot_date, run_mode)
        schema_records.append({
            "run_id": run_id,
            "semantic_model": model_name,
            "source_table": table_name,
            "schema_hash": schema_hash,
            "columns_json": columns_json,
            "recorded_utc": extracted_at,
        })
    except Exception as error:
        status = "Failed"
        error_message = f"{type(error).__name__}: {error}"[:4000]
        failures.append(f"{model_name}.{table_name}: {error_message}")

    audit_records.append({
        "run_id": run_id,
        "run_mode": run_mode,
        "semantic_model": model_name,
        "source_table": table_name,
        "target_table": target_table,
        "source_row_count": source_count,
        "extracted_row_count": extracted_count,
        "schema_hash": schema_hash,
        "status": status,
        "error_message": error_message,
        "started_utc": started,
        "ended_utc": datetime.now(timezone.utc),
    })

run_audit_path = f"{control_root}/run_audit"
schema_history_path = f"{control_root}/schema_history"
run_audit_schema = spark.read.format("delta").load(run_audit_path).schema
schema_history_schema = spark.read.format("delta").load(schema_history_path).schema
spark.createDataFrame(audit_records, run_audit_schema).write.format("delta").mode("append").save(run_audit_path)
if schema_records:
    spark.createDataFrame(schema_records, schema_history_schema).write.format("delta").mode("append").save(schema_history_path)

summary = {
    "run_id": run_id,
    "run_mode": run_mode,
    "table_count": len(audit_records),
    "succeeded": sum(1 for row in audit_records if row["status"] == "Succeeded"),
    "failed": len(failures),
    "snapshot_date_utc": snapshot_date.isoformat(),
}
display(spark.createDataFrame(audit_records, run_audit_schema))
if failures:
    raise RuntimeError("STEP 02 FAILED\n" + "\n".join(failures))

print("STEP 02 PASSED")
print(json.dumps(summary, indent=2))
exit_value = json.dumps(summary, separators=(",", ":"))
notebookutils.notebook.exit(exit_value)
